# Glucose-Level Food Recommender: Train and Export as JSON
Trains the final model on all transactions and exports it as JSON.

### Step 1: Install, import and settings

In [1]:
%pip install -q pyECLAT

In [2]:
import os, sys, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, spearmanr

from pyECLAT import ECLAT
from mlxtend.frequent_patterns import apriori, fpgrowth
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

In [3]:
# SETTINGS
DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive/NHANES")
TRANSACTIONS_FILE = os.path.join(DATA_DIR, "glucose_food_training_data_v3.csv")
NUTRITION_FILE    = os.path.join(DATA_DIR, "food_group_nutrition.csv")
ITEMS_FILE        = os.path.join(DATA_DIR, "food_group_items.csv")
ITEM_BASKETS_FILE = os.path.join(DATA_DIR, "food_group_item_baskets.csv")

SEED = 42
MIN_SUPPORT = 0.08
MIN_PEOPLE = 10
MIN_POOL = 80
TOP_K = 10
N_PAIRS = 5

# Glucose levels
LEVEL_START, LEVEL_STEP, LEVEL_TOP = 80, 20, 200
LEVEL_EDGES = [0] + list(range(LEVEL_START, LEVEL_TOP + 1, LEVEL_STEP)) + [np.inf]
LEVELS = ([f"<{LEVEL_START}"] + [f"{a}-{a + LEVEL_STEP - 1}" for a in range(LEVEL_START, LEVEL_TOP, LEVEL_STEP)]
          + [f"{LEVEL_TOP}+"])

# Nutrition limits per level
def level_limits(i):
    if i <= 1:
        return dict(sugar=15, net_carb=60, sat_fat=10, carb_penalty=0.0)
    k = i - 1
    return dict(sugar=13 - k, net_carb=52 - 7 * k, sat_fat=10 if k <= 2 else 8, carb_penalty=round(0.15 * k, 2))
LIMITS = {lv: level_limits(i) for i, lv in enumerate(LEVELS)}

FIBER_EXCEPTION = dict(min_fiber=1.5, max_sugar=12)       # whole, fibre-rich foods (e.g. fruit) may exceed the sugar limit
NEVER = {953, 924, 925, 929, 923,                          # sports/energy drinks, soft drinks, fruit drinks, sweet tea
         581, 241, 275, 263, 714, 712, 544, 754, 521}      # pizza, nuggets, burgers, fried shrimp, fries, chips, snacks, onion rings, biscuits
NEVER_PREFIX = {93, 81, 82, 83}                            # alcohol (930-939); butter, oils, dressings (810-839)
FROM_100 = {612, 641, 115, 252}                            # from 100 mg/dL: fruit juices, chocolate milk, deli/processed meats

# Clinical label shown next to the level
def clinical_band(g):
    return "Low (hypoglycemia)" if g < 70 else "Normal" if g < 100 else "Prediabetes" if g < 126 else "Diabetes"

# "people like you"
AGE_CUTS, AGE_GROUPS = [17, 39, 59, 200], ["18-39", "40-59", "60+"]

print(pd.DataFrame(LIMITS).T.rename_axis("glucose level (mg/dL)"))

                       sugar  net_carb  sat_fat  carb_penalty
glucose level (mg/dL)                                        
<80                     15.0      60.0     10.0          0.00
80-99                   15.0      60.0     10.0          0.00
100-119                 12.0      45.0     10.0          0.15
120-139                 11.0      38.0     10.0          0.30
140-159                 10.0      31.0      8.0          0.45
160-179                  9.0      24.0      8.0          0.60
180-199                  8.0      17.0      8.0          0.75
200+                     7.0      10.0      8.0          0.90


### Step 2: Load the data

In [4]:
if "google.colab" in sys.modules and not os.environ.get("DATA_DIR"):
    from google.colab import drive
    drive.mount("/content/drive")

data = pd.read_csv(TRANSACTIONS_FILE)
NUT = pd.read_csv(NUTRITION_FILE).set_index("code")
food_cols = [c for c in data.columns if c not in ["SEQN", "Glucose_mg_dL", "Age", "Gender"]]

def level_of(glucose):
    return pd.cut(pd.Series(glucose, dtype=float), LEVEL_EDGES, labels=LEVELS, right=False).astype(str)

def age_group_of(age):
    return pd.cut(pd.Series(age, dtype=float), AGE_CUTS, labels=AGE_GROUPS).astype(str)

# Pipeline step 1
class AddLevelAndGroup(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["Level"] = level_of(X.Glucose_mg_dL).values
        X["Group"] = (age_group_of(X.Age) + " " + X.Gender.str.lower()).values
        return X

data = AddLevelAndGroup().fit_transform(data)
GROUPS = sorted(data.Group.unique())

def pretty(col): return col.rsplit("_", 1)[0].replace("_", " ")
def code_of(col): return int(col.rsplit("_", 1)[1])

print("Transactions:", len(data), "| items (food groups):", len(food_cols), "| nutrition rows:", len(NUT))

Mounted at /content/drive
Transactions: 1885 | items (food groups): 184 | nutrition rows: 184


### Step 3: Model functions

In [5]:
# ECLAT helpers
def to_eclat_format(onehot_rows):
    return pd.DataFrame([list(onehot_rows.columns[row.astype(bool)]) for row in onehot_rows.values])

def min_support_for(n): return max(MIN_SUPPORT, MIN_PEOPLE / n)

def wilson_ci(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half

def bh(p):
    p = np.asarray(p, float); n = len(p)
    if n == 0: return p
    order = np.argsort(p); q = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(q, 1); return out

def mine_group(rows, everyone, max_combination=2):
    n = len(rows)
    eclat = ECLAT(data=to_eclat_format(rows), verbose=False)
    _, supports = eclat.fit(min_support=min_support_for(n), min_combination=1,
                            max_combination=max_combination, separator=" & ", verbose=False)
    others = everyone.drop(index=rows.index, errors="ignore")
    single = rows.mean()
    recs = []
    for key, sup in supports.items():
        items = tuple(key.split(" & ")); k = int(round(sup * n))
        if len(items) == 1:
            f = items[0]; k_out = int(others[f].sum()); n_out = len(others)
            lift = sup / ((k + k_out) / (n + n_out))
            p = fisher_exact([[k, n - k], [k_out, n_out - k_out]])[1]
        else:
            a, b = items; ka, kb = int(rows[a].sum()), int(rows[b].sum())
            lift = sup / (single[a] * single[b])
            p = fisher_exact([[k, ka - k], [kb - k, n - ka - kb + k]])[1]
        lo, hi = wilson_ci(k, n)
        recs.append(dict(items=items, size=len(items), count=k, support=sup, ci_low=lo, ci_high=hi, lift=lift, p=p))
    t = pd.DataFrame(recs, columns=["items", "size", "count", "support", "ci_low", "ci_high", "lift", "p"])
    t["q"] = bh(t.p.values)
    return t

def make_pools(frame, min_people=MIN_POOL):
    counts = frame.Level.value_counts().reindex(LEVELS, fill_value=0)
    mid = int(np.argmax(counts.values))
    pool_of = {LEVELS[mid]: (LEVELS[mid],)}
    for side in (LEVELS[:mid], LEVELS[mid + 1:][::-1]):
        current, last = [], None
        for lv in side:
            current.append(lv)
            if counts[current].sum() >= min_people:
                pool = tuple(sorted(current, key=LEVELS.index))
                for l in current: pool_of[l] = pool
                current, last = [], pool
        if current:
            target = last if last else pool_of[LEVELS[mid]]
            pool = tuple(sorted(set(target) | set(current), key=LEVELS.index))
            for l in pool: pool_of[l] = pool
    return pool_of

def pool_name(pool):
    lo, hi = LEVEL_EDGES[LEVELS.index(pool[0])], LEVEL_EDGES[LEVELS.index(pool[-1]) + 1]
    if lo == 0 and np.isinf(hi): return "all levels"
    if lo == 0: return f"below {int(hi)}"
    if np.isinf(hi): return f"{int(lo)} and above"
    return f"{int(lo)}-{int(hi) - 1}"

# Nutrition filter
def is_fit(col, level):
    c = code_of(col)
    if c not in NUT.index or c in NEVER or c // 10 in NEVER_PREFIX: return False
    if LEVELS.index(level) >= LEVELS.index("100-119") and c in FROM_100: return False
    n, lim = NUT.loc[c], LIMITS[level]
    sugar_ok = n.sugar_100g <= lim["sugar"] or (n.fiber_100g >= FIBER_EXCEPTION["min_fiber"] and n.sugar_100g <= FIBER_EXCEPTION["max_sugar"])
    return bool(sugar_ok and n.net_carb_100g <= lim["net_carb"] and n.sat_fat_100g <= lim["sat_fat"])

FIT = {lv: {c for c in food_cols if is_fit(c, lv)} for lv in LEVELS}

# The model
def train_model(frame, pairs=True):
    pool_of = make_pools(frame)
    pool_tables = {pool: mine_group(frame.loc[frame.Level.isin(pool), food_cols], frame[food_cols], 2 if pairs else 1)
                   for pool in sorted(set(pool_of.values()), key=lambda p: LEVELS.index(p[0]))}
    group_tables = {g: mine_group(frame.loc[frame.Group == g, food_cols], frame[food_cols], 1) for g in GROUPS}
    sizes = {"pool": {p: int(frame.Level.isin(p).sum()) for p in pool_tables}, "group": frame.Group.value_counts().to_dict()}
    return dict(pool_of=pool_of, pool_tables=pool_tables, group_tables=group_tables, sizes=sizes)

def singles(t):
    s = t[t["size"] == 1]
    return {it[0]: r for it, r in zip(s["items"], s.to_dict("records"))}

def recommend_foods(model, level, group, k=TOP_K):
    pt = singles(model["pool_tables"][model["pool_of"][level]]); gt = singles(model["group_tables"][group])
    w = LIMITS[level]["carb_penalty"]; rows = []
    for f in (set(pt) | set(gt)) & FIT[level]:
        ls, gs = pt.get(f, {}).get("support", 0.0), gt.get(f, {}).get("support", 0.0)
        support = (ls + gs) / 2
        nc = NUT.loc[code_of(f), "net_carb_100g"]
        rows.append(dict(food=f, score=support * (1 - w * min(nc / 40, 1)), support=support,
                         level_support=ls, group_support=gs,
                         level_lift=pt.get(f, {}).get("lift", np.nan), level_q=pt.get(f, {}).get("q", np.nan),
                         net_carb_100g=nc))
    return pd.DataFrame(rows).sort_values(["score", "support"], ascending=False).head(k).reset_index(drop=True)

def recommend_pairs(model, level, k=N_PAIRS):
    t = model["pool_tables"][model["pool_of"][level]]
    t = t[(t["size"] == 2) & (t.lift > 1) & (t.q < 0.05)]
    t = t[t["items"].apply(lambda it: all(i in FIT[level] for i in it))]
    return t.sort_values("support", ascending=False).head(k).reset_index(drop=True)

In [6]:
def show_foods(r):
    return pd.DataFrame({"food": r.food.map(pretty), "score": r.score.round(3),
                         "support % (avg)": (r.support * 100).round(1),
                         "% at your level": (r.level_support * 100).round(1),
                         "% your age+gender": (r.group_support * 100).round(1),
                         "lift (level)": r.level_lift.round(2), "q": r.level_q.round(3),
                         "net carb g/100g": r.net_carb_100g.round(1)}).to_string(index=False)

def show_pairs(t):
    if len(t) == 0: return "none for this level"
    return pd.DataFrame({"pair": [" + ".join(pretty(i) for i in it) for it in t["items"]],
                         "support %": (t.support * 100).round(1),
                         "95% CI": [f"{a*100:.0f}-{b*100:.0f}%" for a, b in zip(t.ci_low, t.ci_high)],
                         "lift": t.lift.round(2), "q": t.q.round(3)}).to_string(index=False)

def recommend(model, glucose_mg_dl, age, gender, k=TOP_K):
    gender = str(gender).strip().capitalize()
    if gender not in ("Male", "Female"): raise ValueError("gender must be 'male' or 'female'")
    if age < 18: raise ValueError("the model was built on adults (18+)")
    if not 20 <= glucose_mg_dl <= 600: raise ValueError("glucose must be between 20 and 600 mg/dL")
    level = level_of(glucose_mg_dl)[0]; group = f"{age_group_of(age)[0]} {gender.lower()}"
    pool = model["pool_of"][level]
    print(f"Fasting glucose {glucose_mg_dl} mg/dL -> level {level} mg/dL ({clinical_band(glucose_mg_dl)})")
    print(f"People like you: {group} ({model['sizes']['group'][group]} transactions) | "
          f"same glucose level: {pool_name(pool)} mg/dL ({model['sizes']['pool'][pool]} transactions)")
    print(f"Nutrition limits for this level (per 100 g): sugar <= {LIMITS[level]['sugar']} g, "
          f"net carb <= {LIMITS[level]['net_carb']} g, sat. fat <= {LIMITS[level]['sat_fat']} g, carb penalty {LIMITS[level]['carb_penalty']}")
    print("\nRecommended foods:"); print(show_foods(recommend_foods(model, level, group, k)))
    print("\nGood combinations (fit pairs eaten together significantly more than chance):")
    print(show_pairs(recommend_pairs(model, level)))
    if glucose_mg_dl < 70:
        print("\nNote: below 70 mg/dL is low blood glucose. Seek medical advice.")
    elif glucose_mg_dl >= 100:
        print("\nNote: a fasting glucose in this range should be confirmed and discussed with a doctor.")

### Step 3b: The scikit-learn Pipeline

In [7]:
# Pipeline step 2
class ECLATRecommender(BaseEstimator):
    def __init__(self, pairs=True, top_k=TOP_K):
        self.pairs = pairs
        self.top_k = top_k

    def fit(self, X, y=None):
        self.model_ = train_model(X, pairs=self.pairs)
        return self

    def predict(self, X):
        top = {}
        for (lv, g), people in X.groupby(["Level", "Group"]):
            foods = list(recommend_foods(self.model_, lv, g, self.top_k).food)
            for i in people.index:
                top[i] = foods
        return pd.Series(top).reindex(X.index)

    def _hits(self, X):
        top = self.predict(X)
        return pd.Series([X.loc[i, top[i]].sum() for i in X.index], index=X.index)

    def hit_rate(self, X, y=None):
        return float((self._hits(X) > 0).mean())

    def precision(self, X, y=None):
        return float((self._hits(X) / self.top_k).mean())

    def score(self, X, y=None):
        return self.hit_rate(X)

# The Pipeline
def make_sucrose_pipeline(pairs=True):
    return Pipeline([("levels_groups", AddLevelAndGroup()),
                     ("eclat", ECLATRecommender(pairs=pairs))])

print(make_sucrose_pipeline())

Pipeline(steps=[('levels_groups', AddLevelAndGroup()),
                ('eclat', ECLATRecommender())])


### Step 4: Train the final model on all transactions

In [8]:
t0 = time.time()
sucrose_pipeline = make_sucrose_pipeline().fit(data)
model = sucrose_pipeline.named_steps["eclat"].model_
print(f"ECLAT finished in {time.time() - t0:.0f} s")

ECLAT finished in 182 s


### Step 5: Precompute every answer and build the JSON

In [9]:
def food_records(r):
    return [{"food": pretty(f), "code": code_of(f), "score": round(float(s), 4), "support": round(float(sp), 4),
             "support_at_level": round(float(ls), 4), "support_age_gender": round(float(gs), 4),
             "lift_level": None if pd.isna(l) else round(float(l), 3), "q_level": None if pd.isna(q) else round(float(q), 4),
             "net_carb_100g": round(float(nc), 1)}
            for f, s, sp, ls, gs, l, q, nc in zip(r.food, r.score, r.support, r.level_support, r.group_support,
                                                  r.level_lift, r.level_q, r.net_carb_100g)]

def pair_records(t):
    return [{"foods": [pretty(i) for i in it], "codes": [code_of(i) for i in it], "support": round(float(s), 4),
             "ci_95": [round(float(a), 4), round(float(b), 4)], "lift": round(float(l), 3), "q": round(float(q), 4)}
            for it, s, a, b, l, q in zip(t["items"], t.support, t.ci_low, t.ci_high, t.lift, t.q)]

export = {
    "model": "glucose-level-food-eclat",
    "version": "2.1",
    "algorithm": "pyECLAT frequent itemsets (support, lift, Fisher/Benjamini-Hochberg q) + nutrition filter per level",
    "trained_on": f"NHANES 2017-2018, {len(data)} adults without diagnosed diabetes",
    "glucose_levels": [{"level": lv, "min_mg_dl": float(LEVEL_EDGES[i]),
                        "max_mg_dl": None if np.isinf(LEVEL_EDGES[i + 1]) else float(LEVEL_EDGES[i + 1]),
                        "limits_per_100g": LIMITS[lv],
                        "mined_with": pool_name(model["pool_of"][lv])} for i, lv in enumerate(LEVELS)],
    "clinical_bands": [{"band": "Low (hypoglycemia)", "min_mg_dl": 0, "max_mg_dl": 70},
                       {"band": "Normal", "min_mg_dl": 70, "max_mg_dl": 100},
                       {"band": "Prediabetes", "min_mg_dl": 100, "max_mg_dl": 126},
                       {"band": "Diabetes", "min_mg_dl": 126, "max_mg_dl": None}],
    "age_groups": [{"group": "18-39", "min_age": 18, "max_age": 39}, {"group": "40-59", "min_age": 40, "max_age": 59},
                   {"group": "60+", "min_age": 60, "max_age": None}],
    "fiber_exception": FIBER_EXCEPTION,
    "input_rules": {"min_age": 18, "genders": ["male", "female"], "glucose_range_mg_dl": [20, 600]},
    "notes": {"low": "Below 70 mg/dL is low blood glucose. Seek medical advice.",
              "high": "A fasting glucose of 100 mg/dL or more should be confirmed and discussed with a doctor.",
              "disclaimer": "Educational prototype, not medical advice."},
    "recommendations": {lv: {g: {"people_at_level": model["sizes"]["pool"][model["pool_of"][lv]],
                                 "people_in_group": int(model["sizes"]["group"][g]),
                                 "foods": food_records(recommend_foods(model, lv, g)),
                                 "pairs": pair_records(recommend_pairs(model, lv))}
                             for g in GROUPS} for lv in LEVELS},
}

# The specific foods inside every recommended group
ITEMS = pd.read_csv(ITEMS_FILE)
BASKETS = pd.read_csv(ITEM_BASKETS_FILE)
ate = (pd.crosstab(BASKETS.SEQN, BASKETS.code).clip(upper=1)
         .reindex(data.SEQN.values, fill_value=0).set_axis(data.index))
item_level_support = {lv: ate[data.Level.isin(model["pool_of"][lv]).values].mean() for lv in LEVELS}
item_group_support = {g: ate[(data.Group == g).values].mean() for g in GROUPS}

def item_record(r):
    c = int(r.code)
    return {"code": c, "name": str(r.name), "people": int(r.people),
            "net_carb_100g": round(float(r.net_carb_100g), 1), "sugar_100g": round(float(r.sugar_100g), 1),
            "fiber_100g": round(float(r.fiber_100g), 1), "sat_fat_100g": round(float(r.sat_fat_100g), 1),
            "support_at_level": {lv: round(float(item_level_support[lv].get(c, 0.0)), 4) for lv in LEVELS},
            "support_age_gender": {g: round(float(item_group_support[g].get(c, 0.0)), 4) for g in GROUPS}}

recommended = sorted({f["code"] for lv in export["recommendations"].values() for g in lv.values() for f in g["foods"]})
export["group_items"] = {
    str(c): [item_record(r) for r in ITEMS[ITEMS.group == c].sort_values(["people", "code"], ascending=[False, True]).itertuples()]
    for c in recommended}
export["item_ranking"] = ("score = (support_at_level + support_age_gender) / 2 x (1 - carb_penalty x min(net_carb_100g / 40, 1)), "
                          "highest first; only foods within the level's limits_per_100g (with fiber_exception) are shown")
print("Specific foods exported for", len(export["group_items"]), "recommended groups:",
      sum(len(v) for v in export["group_items"].values()), "foods")

OUT_JSON = os.path.join(os.environ.get("OUT_DIR", DATA_DIR), "glucose_food_model.json")
with open(OUT_JSON, "w") as f:
    json.dump(export, f, indent=2)
print("Saved:", OUT_JSON, f"({os.path.getsize(OUT_JSON) / 1024:.0f} KB)")

Specific foods exported for 20 recommended groups: 233 foods
Saved: /content/drive/MyDrive/NHANES/glucose_food_model.json (371 KB)


### Step 6: Check the JSON works without the model

In [10]:
with open(OUT_JSON) as f:
    M = json.load(f)

def lookup(M, glucose, age, gender):
    gender = gender.strip().lower()
    r = M["input_rules"]
    if gender not in r["genders"] or age < r["min_age"] or not r["glucose_range_mg_dl"][0] <= glucose <= r["glucose_range_mg_dl"][1]:
        raise ValueError("invalid input")
    level = next(l["level"] for l in M["glucose_levels"] if glucose >= l["min_mg_dl"] and (l["max_mg_dl"] is None or glucose < l["max_mg_dl"]))
    age_group = next(a["group"] for a in M["age_groups"] if age >= a["min_age"] and (a["max_age"] is None or age <= a["max_age"]))
    band = next(b["band"] for b in M["clinical_bands"] if glucose >= b["min_mg_dl"] and (b["max_mg_dl"] is None or glucose < b["max_mg_dl"]))
    rec = M["recommendations"][level][f"{age_group} {gender}"]
    return {"level": level, "clinical_band": band, "group": f"{age_group} {gender}", **rec}

ok = True
for glu in [65, 92, 105, 126, 150, 175, 190, 250]:
    for age in [25, 45, 70]:
        for gen in ["male", "female"]:
            out = lookup(M, glu, age, gen)
            ref = [pretty(f) for f in recommend_foods(model, level_of(glu)[0], f"{age_group_of(age)[0]} {gen}").food]
            ok &= [x["food"] for x in out["foods"]] == ref
print("JSON lookups identical to the model:", ok)
res = lookup(M, 150, 67, "female")
print(res["level"], "|", res["clinical_band"], "|", res["group"])
print([x["food"] for x in res["foods"]])
print([" + ".join(p["foods"]) for p in res["pairs"]])

JSON lookups identical to the model: True
140-159 | Diabetes | 60+ female
['Fresh fruit', 'Milk', 'Salad vegetables', 'Cooked vegetables', 'Rice and oatmeal', 'Eggs boiled or fried', 'Raw tomatoes', 'Fish', 'Nuts', 'Scrambled eggs and omelets']
['Salad vegetables + Raw tomatoes']


### Step 7: Download

In [11]:
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(OUT_JSON)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>